In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class MyCallback(BaseCallbackHandler):

    def on_chain_start(self, serialized, inputs, **kwargs):
        print("[CHAIN] START")

    def on_chain_end(self, outputs, **kwargs):
        print("[CHAIN] END")

    def on_tool_start(self, serialized, input_str, **kwargs):
        print("[TOOL] START")
        print("Input:", input_str)

    def on_tool_end(self, output, **kwargs):
        print("[TOOL] END")
        print("Output:", output)

    def on_llm_start(self, serialized, prompts, **kwargs):
        print("[LLM] START")

    def on_llm_end(self, response, **kwargs):
        print("[LLM] END")

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler

# خواندن متغیرهای .env
load_dotenv()
model_name = os.getenv("OLLAMA_MODEL")
if not model_name:
    raise ValueError("OLLAMA_MODEL در فایل .env تعریف نشده است.")

# =====================================================
# 1. Callback
# =====================================================

class MyCallback(BaseCallbackHandler):

    def on_tool_start(self, serialized, input_str, **kwargs):
        print("\n" + "=" * 50)
        print("[CALLBACK] TOOL START")
        print("Tool:", serialized.get("name"))
        print("Input:", input_str)
        print("=" * 50)

    def on_tool_end(self, output, **kwargs):
        print("\n[CALLBACK] TOOL END")
        print("Output:", output)

    def on_llm_start(self, serialized, prompts, **kwargs):
        print("\n[CALLBACK] LLM START")

    def on_llm_end(self, response, **kwargs):
        print("[CALLBACK] LLM END")


# =====================================================
# 2. Tool
# =====================================================

@tool
def get_weather(city: str) -> str:
    """
    دمای یک شهر را برمی‌گرداند.
    """

    weather = {
        "تهران": "25 درجه",
        "مشهد": "20 درجه",
        "تبریز": "15 درجه",
    }

    return weather.get(
        city.lower(),
        "اطلاعاتی برای این شهر ندارم."
    )


# =====================================================
# 3. Model
# =====================================================

model = init_chat_model(
    model=model_name,
    model_provider="ollama",
    temperature=0,
)


# =====================================================
# 4. Agent
# =====================================================

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="""
تو یک دستیار مفید هستی.

اگر کاربر درباره آب‌وهوا سؤال کرد،
از ابزار get_weather استفاده کن.

پاسخ را به فارسی بده.
""",
)


# =====================================================
# 5. Run
# =====================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "دمای مشهد چقدر است؟",
            }
        ]
    },
    config={
        "callbacks": [
            MyCallback()
        ]
    }
)


# =====================================================
# 6. Final Answer
# =====================================================

print("\n" + "#" * 60)
print("FINAL ANSWER")
print("#" * 60)

print(result["messages"][-1].content)


[CALLBACK] LLM START
[CALLBACK] LLM END

[CALLBACK] TOOL START
Tool: get_weather
Input: {'city': 'مشهد'}

[CALLBACK] TOOL END
Output: content='20 درجه' name='get_weather' tool_call_id='27d8bf8e-958c-454e-8410-ac16d240895f'

[CALLBACK] LLM START
[CALLBACK] LLM END

############################################################
FINAL ANSWER
############################################################
دمای مشهد در حال حاضر ۲۰ درجه است.
